# Stochastic Calculus, Step by Step · Lesson 32: Ordinary differential equations

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. For interest rates this lesson uses Lesson 28's Vasicek model: speed κ = 0.5, long-run level r̄ = 4%, volatility s = 1% and today's rate r₀ = 3%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. The integrating factor · 3. Riccati equations and the CIR bond · 4. Solving numerically: Euler and Runge-Kutta · 5. When the solution blows up · 6. Exercises · 7. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 32,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

## 2. The integrating factor

Lesson 28's Vasicek rate, $dr = \kappa(\bar r - r)\,dt + s\,dW$. Its average $m(t) = E[r_t]$ solves $m' = \kappa(\bar r - m)$ (take expectations; the $dW$ term has mean 0), and its variance solves $v' = -2\kappa v + s^2$. Both are first-order linear equations, solved by multiplying by $e^{\kappa t}$ (or $e^{2\kappa t}$). We check the formulas against simulated paths.

In [ ]:
kappa, rbar, s, r0 = 0.5, 0.04, 0.01, 0.03
m_ = lambda t: rbar + (r0 - rbar) * math.exp(-kappa * t)
v_ = lambda t: s**2 * (1 - math.exp(-2 * kappa * t)) / (2 * kappa)
B_vas = lambda tau: (1 - math.exp(-kappa * tau)) / kappa
A_vas = lambda tau: (rbar - s**2 / (2 * kappa**2)) * (B_vas(tau) - tau) - s**2 * B_vas(tau) ** 2 / (4 * kappa)

rng = np.random.default_rng(seed)
dt = 1 / 252
paths, rr = 20_000, np.full(20_000, r0)
marks, rows, step = {0.5: None, 1.0: None, 2.0: None, 5.0: None}, [], 0
for n in range(1, int(5 / dt) + 1):
    rr = rr + kappa * (rbar - rr) * dt + s * math.sqrt(dt) * rng.standard_normal(paths)
    t_ = round(n * dt, 6)
    for tm in marks:
        if abs(t_ - tm) < dt / 2:
            rows.append({"t": tm, "mean": m_(tm), "mean_sim": float(rr.mean()), "sd": math.sqrt(v_(tm)), "sd_sim": float(rr.std())})
out["ifac"] = {"kappa": kappa, "rbar": rbar, "s": s, "r0": r0, "rows": rows, "half_life": math.log(2) / kappa,
               "sd_long": s / math.sqrt(2 * kappa), "B": {f"t{int(t)}": B_vas(t) for t in (1, 5, 10)}}
# a direction field for m' = kappa (rbar - m), and solutions from several starts
tt = np.linspace(0, 8, 81)
out["ifac"]["fig"] = {"t": tt.round(3).tolist(),
                      **{f"m{int(x * 1000)}": (rbar + (x - rbar) * np.exp(-kappa * tt)).round(6).tolist() for x in (0.0, 0.03, 0.06, 0.08)}}
pd.DataFrame(rows).round(5)

## 3. Riccati equations and the CIR bond

In the **CIR model** the rate's volatility is $s_c\sqrt r$, so the rate can't go negative: $dr = \kappa(\bar r - r)\,dt + s_c\sqrt r\,dW$. Take $s_c$ so that today's volatility matches Vasicek's: $s_c\sqrt{r_0} = 1\%$. The bond's affine guess $P = e^{A - Br}$ now gives an equation for $B$ with a square in it, a **Riccati equation**:

$$B' = 1 - \kappa B - \tfrac12s_c^2B^2, \qquad A' = -\kappa\bar rB, \qquad A(0) = B(0) = 0 .$$

It has a closed form. We compare CIR and Vasicek yields, and check one CIR bond by simulation.

In [ ]:
sc = s / math.sqrt(r0)
gam = math.sqrt(kappa**2 + 2 * sc**2)


def B_cir(tau):
    e = math.exp(gam * tau) - 1
    return 2 * e / ((gam + kappa) * e + 2 * gam)


def A_cir(tau):
    e = math.exp(gam * tau) - 1
    return (2 * kappa * rbar / sc**2) * math.log(2 * gam * math.exp((gam + kappa) * tau / 2) / ((gam + kappa) * e + 2 * gam))


P_cir = lambda tau, x=r0: math.exp(A_cir(tau) - B_cir(tau) * x)
P_vas = lambda tau, x=r0: math.exp(A_vas(tau) - B_vas(tau) * x)
y_rows = [{"tau": tau, "B_cir": B_cir(tau), "B_vas": B_vas(tau), "y_cir": -math.log(P_cir(tau)) / tau * 100, "y_vas": -math.log(P_vas(tau)) / tau * 100}
          for tau in (1, 2, 5, 10, 30)]
# the Riccati right-hand side vanishes at B_inf: the long-maturity limit of B
B_inf = (gam - kappa) / sc**2
# simulation check of the 5-year CIR bond: full-truncation Euler, trapezium integral of r
rng = np.random.default_rng(seed + 1)
h, n_steps, M = 1 / 100, 500, 50_000
x = np.full(M, r0); integral = np.zeros(M)
for _ in range(n_steps):
    xp = np.maximum(x, 0)
    x_new = x + kappa * (rbar - xp) * h + sc * np.sqrt(xp * h) * rng.standard_normal(M)
    integral += 0.5 * (np.maximum(x, 0) + np.maximum(x_new, 0)) * h
    x = x_new
disc_ = np.exp(-integral)
tt = np.linspace(0.25, 30, 120)
out["cir"] = {"sc": sc, "gamma": gam, "rows": y_rows, "B_inf": B_inf, "B_vas_inf": 1 / kappa,
              "mc5": float(disc_.mean()), "mc5_se": float(disc_.std() / math.sqrt(M)), "P5": P_cir(5), "P5_vas": P_vas(5),
              "fig": {"tau": tt.round(3).tolist(), "y_cir": [-math.log(P_cir(t)) / t * 100 for t in tt],
                      "y_vas": [-math.log(P_vas(t)) / t * 100 for t in tt]}}
pd.DataFrame(y_rows).round(5)

## 4. Solving numerically: Euler and Runge-Kutta

Most ODEs in finance have no closed form. **Euler's method** steps along the slope at the start of each step; **Runge-Kutta 4** averages four slopes taken inside the step. Solve the CIR pair $(A, B)$ out to 10 years with each, and compare $B(10)$ with the closed form as the step $h$ shrinks.

In [ ]:
def rhs(y):
    A_, B_ = y
    return np.array([-kappa * rbar * B_, 1 - kappa * B_ - 0.5 * sc**2 * B_**2])


def euler(f, y0, t_end, h):
    y = np.array(y0, float)
    for _ in range(round(t_end / h)):
        y = y + h * f(y)
    return y


def rk4(f, y0, t_end, h):
    y = np.array(y0, float)
    for _ in range(round(t_end / h)):
        k1 = f(y); k2 = f(y + h / 2 * k1); k3 = f(y + h / 2 * k2); k4 = f(y + h * k3)
        y = y + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return y


exact10 = B_cir(10)
rk_rows = []
for h_ in (2.0, 1.0, 0.5, 0.25, 0.125):
    pe = euler(rhs, (0, 0), 10, h_); pr = rk4(rhs, (0, 0), 10, h_)
    rk_rows.append({"h": h_, "steps": round(10 / h_), "euler": pe[1] - exact10, "rk4": pr[1] - exact10})
ratios = {"euler": rk_rows[-2]["euler"] / rk_rows[-1]["euler"], "rk4": rk_rows[-2]["rk4"] / rk_rows[-1]["rk4"]}
pr = rk4(rhs, (0, 0), 10, 0.5)
out["rk"] = {"exact10": exact10, "rows": rk_rows, "ratio_euler": ratios["euler"], "ratio_rk4": ratios["rk4"],
             "P10_rk4_h05": math.exp(pr[0] - pr[1] * r0), "P10": P_cir(10)}
pd.DataFrame(rk_rows)

## 5. When the solution blows up

A Riccati equation with the "wrong" sign can run off to infinity in finite time: $y' = 1 + y^2$, $y(0) = 0$, has the solution $\tan t$, which explodes at $t = \pi/2$. A fixed-step solver doesn't notice: it just returns large numbers. A simple check: take each step once with $h$ and twice with $h/2$, and compare.

In [ ]:
g = lambda y: 1 + y**2
h_ = 0.01
y_ = 0.0; t_ = 0.0; blow_rows = []; marks = (0.5, 1.0, 1.5, 1.55, 1.56, 1.57)
while t_ < 1.57 - 1e-9:
    k1 = g(y_); k2 = g(y_ + h_ / 2 * k1); k3 = g(y_ + h_ / 2 * k2); k4 = g(y_ + h_ * k3)
    big = y_ + h_ / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    half = y_
    for _ in range(2):
        a1 = g(half); a2 = g(half + h_ / 4 * a1); a3 = g(half + h_ / 4 * a2); a4 = g(half + h_ / 2 * a3)
        half = half + h_ / 12 * (a1 + 2 * a2 + 2 * a3 + a4)
    est = abs(big - half)
    y_ = half; t_ = round(t_ + h_, 10)
    for m in marks:
        if abs(t_ - m) < 1e-9:
            blow_rows.append({"t": m, "rk4": y_, "tan": math.tan(m), "estimate": est})
out["blow"] = {"rows": blow_rows, "t_star": math.pi / 2}
out["blow"]["fig"] = {"t": np.linspace(0, 1.55, 60).round(4).tolist(), "tan": np.tan(np.linspace(0, 1.55, 60)).round(4).tolist(),
                      "tanh": np.tanh(np.linspace(0, 1.55, 60)).round(4).tolist()}
pd.DataFrame(blow_rows)

In [ ]:
# Chart data for the video (no random numbers)
rkr = out["rk"]["rows"]
out["rk"]["fig"] = {"logh": [round(math.log10(d["h"]), 4) for d in rkr],
                    "euler": [round(math.log10(abs(d["euler"])), 4) for d in rkr],
                    "rk4": [round(math.log10(abs(d["rk4"])), 4) for d in rkr],
                    "ticks": [{"x": round(math.log10(d["h"]), 4), "label": f"{d['h']:g}"} for d in rkr]}
out["blow"]["fig"]["ticks"] = [{"x": v, "label": f"{v:g}"} for v in (0.0, 0.5, 1.0, 1.5)]

## 6. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 5: Solutions*.

**Exercise 1.** Solve $m' = \kappa(\bar r - m)$ with $m(0) = r_0$ by the integrating factor $e^{\kappa t}$. With Lesson 28's numbers ($\kappa = 0.5$, $\bar r = 4\%$, $r_0 = 3\%$), find $E[r_1]$ and $E[r_5]$, and the time it takes to close half the gap to $\bar r$.

**Exercise 2.** The variance $v(t)$ of the Vasicek rate solves $v' = -2\kappa v + s^2$ with $v(0) = 0$. Solve it, find the long-run standard deviation, and evaluate the standard deviation at $t = 1$ with $s = 1\%$.

**Exercise 3.** For the Vasicek bond $P = e^{A(\tau) - B(\tau)r}$, show that $B' = 1 - \kappa B$ and $A' = -\kappa\bar rB + \tfrac12s^2B^2$, with $A(0) = B(0) = 0$. Solve for $B$ with an integrating factor, and find its limit as $\tau \to \infty$.

**Exercise 4.** Show that the substitution $y = -u'/(cu)$ turns the Riccati equation $y' = a + by + cy^2$ into the linear equation $u'' - bu' + acu = 0$. For the CIR equation $B' = 1 - \kappa B - \tfrac12s_c^2B^2$, find the two exponents of $u$, and the limit of $B$ as $\tau \to \infty$.

**Exercise 5.** Solve $y' = y^2$ with $y(0) = 1$, and $y' = 1 + y^2$ with $y(0) = 0$. When does each blow up? Then solve $y' = 1 - y^2$ with $y(0) = 0$, and explain why the sign of the square decides whether the solution stays finite.

**Exercise 6.** Apply Euler's method with step $h$ to $y' = -\kappa y$, $y(0) = 1$. Show that $y_n = (1 - \kappa h)^n$, and compare $y_{10}$ with $e^{-0.5}$ for $\kappa = 0.5$, $h = 0.1$. Show that the error at a fixed time is roughly proportional to $h$. What happens when $h > 2/\kappa$?

**Exercise 7.** (Code) Use Runge-Kutta 4 with $h = 0.1$ to compute the CIR yield curve from 1 to 30 years, and compare it with the closed form. What is the largest error, in basis points?

**Exercise 8.** (Code) Write a step-doubling error estimate for Runge-Kutta 4, apply it to $y' = 1 + y^2$ from $t = 0$ to $t = 1.57$ with $h = 0.01$, and find where the estimate first exceeds $10^{-6}$. How close is that to the blow-up time?

## 7. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")